# Buscador de Papers - Semantic Scholar

**Semantic Scholar** es una plataforma gratuita del Allen Institute for AI con mas de 200 millones de papers.

Ventajas sobre arXiv:
- Cubre **todas las disciplinas** (no solo fisica/CS)
- Muestra **conteo de citas** y referencias
- Detecta si el paper tiene **PDF open access**
- API gratuita sin registro (limite generoso)

---

## Paso 1: Instalar dependencias

In [ ]:
!pip install requests pandas sumy nltk fpdf2 --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('Listo.')

## Paso 2: Importar librerias

In [ ]:
import requests
import pandas as pd
import datetime
import time
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words
from fpdf import FPDF
from google.colab import files
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

BASE_URL = 'https://api.semanticscholar.org/graph/v1'
CAMPOS   = 'title,authors,year,abstract,citationCount,openAccessPdf,externalIds,fieldsOfStudy,url'

print('Librerias importadas.')

## Paso 3: Funciones principales

In [ ]:
def resumir_texto(texto, oraciones=3):
    if not texto or len(texto) < 80:
        return texto or 'Sin abstract disponible.'
    try:
        parser = PlaintextParser.from_string(texto, Tokenizer('english'))
        summarizer = LsaSummarizer(Stemmer('english'))
        summarizer.stop_words = get_stop_words('english')
        return ' '.join(str(s) for s in summarizer(parser.document, oraciones))
    except Exception:
        return texto[:600] + '...' if len(texto) > 600 else texto


def buscar_papers(query, max_results=10, sort='relevance', campo_estudio=None, anio_desde=None):
    """Busca en Semantic Scholar Graph API."""
    params = {
        'query': query,
        'limit': min(max_results, 100),
        'fields': CAMPOS,
    }
    if campo_estudio:
        params['fieldsOfStudy'] = campo_estudio
    if anio_desde:
        params['year'] = f'{anio_desde}-'

    resp = requests.get(f'{BASE_URL}/paper/search', params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json().get('data', [])

    resultados = []
    for p in data:
        abstract = (p.get('abstract') or '').replace('\n', ' ')
        pdf_url   = (p.get('openAccessPdf') or {}).get('url', '')
        doi       = (p.get('externalIds') or {}).get('DOI', 'N/A')
        campos    = ', '.join(p.get('fieldsOfStudy') or [])
        autores   = ', '.join(a.get('name', '') for a in (p.get('authors') or []))
        resultados.append({
            'titulo':    p.get('title', 'Sin titulo'),
            'autores':   autores or 'N/A',
            'anio':      p.get('year', 'N/A'),
            'citas':     p.get('citationCount', 0),
            'campos':    campos or 'N/A',
            'doi':       doi,
            'abstract':  abstract,
            'resumen':   resumir_texto(abstract),
            'url':       p.get('url', ''),
            'pdf_url':   pdf_url,
        })

    # Ordenar por citas si se pide
    if sort == 'citas':
        resultados.sort(key=lambda x: x['citas'], reverse=True)
    elif sort == 'anio':
        resultados.sort(key=lambda x: x['anio'] or 0, reverse=True)

    return resultados


def mostrar_html(resultados):
    if not resultados:
        display(HTML('<p style="color:red">No se encontraron resultados.</p>'))
        return
    html = f'<h3>Se encontraron {len(resultados)} papers:</h3>'
    for i, p in enumerate(resultados, 1):
        pdf_link = f'<a href="{p["pdf_url"]}" target="_blank">PDF Open Access</a>' if p['pdf_url'] else '<span style="color:#aaa">Sin PDF open access</span>'
        html += f'''
        <div style="border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;
                    background:#f9f9f9;font-family:Arial,sans-serif;">
          <h4 style="margin:0 0 6px;color:#1a0dab;">#{i} — {p['titulo']}</h4>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>Autores:</b> {p['autores']}</p>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>Anio:</b> {p['anio']} &nbsp;|
            <b>Citas:</b> {p['citas']} &nbsp;|
            <b>Area:</b> {p['campos']}</p>
          <p style="margin:2px 0;font-size:13px;color:#555;">
            <b>DOI:</b> {p['doi']}</p>
          <details style="margin-top:8px;">
            <summary style="cursor:pointer;color:#1a0dab;font-size:13px;">Ver abstract completo</summary>
            <p style="font-size:12px;color:#333;margin-top:6px;">{p['abstract'] or 'No disponible.'}</p>
          </details>
          <div style="background:#eef6ff;border-left:4px solid #1a73e8;
                      padding:10px;margin-top:10px;border-radius:4px;">
            <b style="font-size:13px;">Resumen automatico:</b>
            <p style="font-size:13px;margin:4px 0;">{p['resumen']}</p>
          </div>
          <p style="margin-top:8px;font-size:13px;">
            <a href="{p['url']}" target="_blank">Ver en Semantic Scholar</a>
            &nbsp;|&nbsp; {pdf_link}
          </p>
        </div>'''
    display(HTML(html))


def exportar_csv(resultados, nombre=None):
    if not resultados:
        print('Sin resultados para exportar.'); return
    nombre = nombre or f'semantic_scholar_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    pd.DataFrame(resultados).to_csv(nombre, index=False, encoding='utf-8-sig')
    print(f'Archivo: {nombre}')
    files.download(nombre)


def exportar_pdf(resultados, nombre=None):
    if not resultados:
        print('Sin resultados para exportar.'); return
    nombre = nombre or f'semantic_scholar_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.pdf'
    safe = lambda t: (t or '').encode('latin-1', 'replace').decode('latin-1')
    pdf = FPDF()
    pdf.set_auto_page_break(True, 15)
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 16)
    pdf.cell(0, 10, 'Resultados - Semantic Scholar', ln=True, align='C')
    pdf.set_font('Helvetica', '', 10)
    pdf.cell(0, 8, f'Generado: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}', ln=True, align='C')
    pdf.ln(6)
    for i, p in enumerate(resultados, 1):
        pdf.set_font('Helvetica', 'B', 12)
        pdf.multi_cell(0, 7, safe(f'{i}. {p["titulo"]}'))
        pdf.set_font('Helvetica', '', 9)
        pdf.multi_cell(0, 6, safe(f'Autores: {p["autores"]}'))
        pdf.multi_cell(0, 6, safe(f'Anio: {p["anio"]}  |  Citas: {p["citas"]}  |  Area: {p["campos"]}'))
        pdf.multi_cell(0, 6, safe(f'DOI: {p["doi"]}  |  URL: {p["url"]}'))
        pdf.ln(2)
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, 'Resumen automatico:', ln=True)
        pdf.set_font('Helvetica', '', 9)
        pdf.multi_cell(0, 6, safe(p['resumen']))
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, 'Abstract:', ln=True)
        pdf.set_font('Helvetica', '', 8)
        pdf.multi_cell(0, 5, safe(p['abstract']))
        pdf.ln(4)
        pdf.set_draw_color(180, 180, 180)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(4)
    pdf.output(nombre)
    print(f'Archivo: {nombre}')
    files.download(nombre)


print('Funciones cargadas.')

## Paso 4: Interfaz interactiva

In [ ]:
CAMPOS_ESTUDIO = [
    ('Todas las areas', ''),
    ('Computer Science', 'Computer Science'),
    ('Medicine', 'Medicine'),
    ('Biology', 'Biology'),
    ('Physics', 'Physics'),
    ('Mathematics', 'Mathematics'),
    ('Engineering', 'Engineering'),
    ('Chemistry', 'Chemistry'),
    ('Psychology', 'Psychology'),
    ('Economics', 'Economics'),
    ('Sociology', 'Sociology'),
    ('Environmental Science', 'Environmental Science'),
]

w_query   = widgets.Text(value='', placeholder='Ej: transformer neural network, CRISPR, climate change...',
                         description='Busqueda:', layout=widgets.Layout(width='70%'),
                         style={'description_width': '80px'})
w_campo   = widgets.Dropdown(options=CAMPOS_ESTUDIO, value='', description='Area:',
                              layout=widgets.Layout(width='48%'), style={'description_width': '80px'})
w_max     = widgets.IntSlider(value=10, min=1, max=100, step=1, description='Cantidad:',
                               layout=widgets.Layout(width='48%'), style={'description_width': '80px'})
w_sort    = widgets.RadioButtons(options=[('Relevancia','relevance'),('Mas citas','citas'),('Mas recientes','anio')],
                                  value='relevance', description='Ordenar:', style={'description_width': '80px'})
w_anio    = widgets.IntText(value=2018, description='Desde anio:', style={'description_width': '90px'},
                             layout=widgets.Layout(width='200px'))
w_filtrar = widgets.Checkbox(value=False, description='Filtrar por anio minimo', indent=False)

btn_buscar = widgets.Button(description='Buscar', button_style='primary', icon='search',
                             layout=widgets.Layout(width='140px', height='36px'))
btn_csv    = widgets.Button(description='CSV', button_style='success', icon='download',
                             layout=widgets.Layout(width='120px', height='36px'), disabled=True)
btn_pdf    = widgets.Button(description='PDF', button_style='warning', icon='file',
                             layout=widgets.Layout(width='120px', height='36px'), disabled=True)

out_status  = widgets.Output()
out_results = widgets.Output()
resultados_globales = []

def on_buscar(b):
    global resultados_globales
    query = w_query.value.strip()
    if not query:
        with out_status: clear_output(); print('Escribe un termino de busqueda.')
        return
    btn_buscar.disabled = True; btn_csv.disabled = True; btn_pdf.disabled = True
    with out_status: clear_output(); print(f'Buscando "{query}"...')
    with out_results: clear_output()
    try:
        anio = w_anio.value if w_filtrar.value else None
        resultados_globales = buscar_papers(query, w_max.value, w_sort.value, w_campo.value or None, anio)
        with out_status: clear_output(); print(f'{len(resultados_globales)} papers encontrados.')
        with out_results: mostrar_html(resultados_globales)
        btn_csv.disabled = False; btn_pdf.disabled = False
    except Exception as e:
        with out_status: clear_output(); print(f'Error: {e}')
    finally:
        btn_buscar.disabled = False

btn_buscar.on_click(on_buscar)
btn_csv.on_click(lambda b: exportar_csv(resultados_globales))
btn_pdf.on_click(lambda b: exportar_pdf(resultados_globales))

display(HTML('<h3 style="font-family:Arial">Buscador de Papers - Semantic Scholar</h3>'))
display(widgets.VBox([
    w_query,
    widgets.HBox([w_campo, w_sort]),
    widgets.HBox([w_max, widgets.VBox([w_filtrar, w_anio])]),
    widgets.HBox([btn_buscar, btn_csv, btn_pdf]),
    out_status, out_results
]))

## Paso 5 (opcional): Busqueda por codigo directo

In [ ]:
QUERY       = 'graph neural networks'  # Termino de busqueda
MAX         = 5                        # Cantidad de papers
SORT        = 'citas'                  # 'relevance', 'citas' o 'anio'
CAMPO       = 'Computer Science'       # Area (dejar '' para todas)
ANIO_DESDE  = 2020                     # Anio minimo (None para sin filtro)

resultados = buscar_papers(QUERY, MAX, SORT, CAMPO, ANIO_DESDE)
mostrar_html(resultados)

# exportar_csv(resultados)
# exportar_pdf(resultados)

---

## Licencias, Politicas de Datos y Uso de la API

### Quien opera Semantic Scholar
Semantic Scholar es desarrollado y mantenido por el **Allen Institute for Artificial Intelligence (AI2)**, una organizacion de investigacion sin fines de lucro fundada por Paul Allen en Seattle, EE.UU. Su mision es democratizar el acceso a la literatura cientifica mediante IA.

---

### Licencia de los datos

| Tipo de dato | Licencia |
|---|---|
| Metadatos de papers (titulo, autores, abstract, citas, anio) | **ODC-BY 1.0** — Open Data Commons Attribution License |
| Open Research Corpus (dataset completo descargable) | **ODC-BY 1.0** — requiere atribucion al usar el dataset |
| Resultados de la API Graph (uso en tiempo real) | **Terminos de uso de Semantic Scholar API** |
| Contenido completo de cada paper (PDF) | Licencia original del autor o editorial |

**ODC-BY 1.0** permite usar, compartir y modificar los datos siempre que se atribuya la fuente. Es compatible con investigacion academica y proyectos sin fines de lucro.

---

### Terminos de uso de la API

- **Gratuita sin registro** para uso academico y de investigacion
- **Limite sin API key:** 100 peticiones cada 5 minutos
- **Con API key gratuita** (registro en el sitio): 1 peticion por segundo
- **Usos permitidos:** investigacion, educacion, prototipos no comerciales
- **Usos que requieren acuerdo comercial:** productos o servicios con fines de lucro que usen los datos a escala
- **Prohibido:** revender los datos de la API, usarlos para entrenar modelos sin autorizacion expresa
- Politica completa: https://www.semanticscholar.org/product/api#api-license

---

### Privacidad y seguridad de datos

- Este notebook **no recopila ni almacena** datos personales del usuario
- Las consultas viajan directamente desde tu entorno (Colab/Kaggle) a los servidores de AI2
- Semantic Scholar puede registrar patrones de uso de la API para control de abusos
- Si registras una API key, esta asociada a tu cuenta de Semantic Scholar (no se guarda en el notebook a menos que tu la escribas)
- Los archivos CSV y PDF generados se descargan localmente; **no se transmiten a servidores externos**

---

### Uso responsable de los datos

Al usar este notebook te comprometes a:
1. Respetar los limites de velocidad de la API
2. No redistribuir los metadatos como base de datos propia sin seguir la licencia ODC-BY
3. Verificar la licencia individual de cada paper antes de reproducir su contenido
4. Citar correctamente los papers en publicaciones academicas usando el DOI proporcionado

---

### Nota de citacion academica

Para citar Semantic Scholar como fuente de datos en publicaciones:
> Semantic Scholar. Allen Institute for Artificial Intelligence. https://www.semanticscholar.org